# XGB classifier — all variables
- Binary classification using all features
- Final model uses data without imputation
- Adult patients only

In [ ]:
# --- standard library ---
import json
import os

# --- numeric / data ---
import numpy as np
import pandas as pd

# --- plotting ---
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import pyplot
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

# --- modelling ---
import xgboost as xgb
from imblearn.over_sampling import RandomOverSampler
from sklearn.metrics import (
    confusion_matrix,
    roc_curve,
    auc,
    accuracy_score,
    recall_score,
    f1_score,
    precision_recall_curve,
    precision_score,
    cohen_kappa_score,
    roc_auc_score,
    average_precision_score,
)
from sklearn.model_selection import (
    GroupShuffleSplit,
    GroupKFold,
    GridSearchCV,
    StratifiedGroupKFold,
)
from xgboost import XGBClassifier

# --- notebook ---
from IPython.core.interactiveshell import InteractiveShell
from IPython.display import display

InteractiveShell.ast_node_interactivity = "all"
pd.set_option('display.max_columns', None)

# Create a truncated version of the Blues colormap
colourmap = mcolors.LinearSegmentedColormap.from_list(
    'Blues_truncated',
    plt.cm.Blues(np.linspace(0.15, 1.0, 256))
)


Initialize metrics dataframe

In [3]:
metrics_df = pd.DataFrame(columns=['data', 'F1_score', 'kappa', 'recall', 'precision', 'accuracy', 'customf1', 'specificity', 'PNV'])

## Functions

In [4]:
def proba_to_labels(p, threshold: float):
    """Convert probabilities to 0/1 labels."""
    return (p >= threshold).astype(int)

In [6]:
def calc_pnv(y_true, y_pred):
    """
    Compute PNV (Negative Predictive Value) for binary data.
    """
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    # Predicted negative = 0
    TN = np.sum((y_pred == 0) & (y_true == 0))  # true negatives
    FN = np.sum((y_pred == 0) & (y_true == 1))  # false negatives

    if TN + FN == 0:
        return np.nan  # no predicted negatives → PNV undefined

    return TN / (TN + FN)

In [7]:
def plot_confusion_matrix_with_percentages(
    y_true,
    y_pred,
    labels=None,
    percent_type="row",   # 'row', 'column', or 'all'
    title=None,
    cmap="viridis",
    threshold=None,
    save_path=None,
    figsize=None,
    text_color=None
):
    from sklearn.metrics import confusion_matrix
    import numpy as np
    import matplotlib.pyplot as plt

    cm = confusion_matrix(y_true, y_pred, labels=labels)
    if labels is None:
        labels = np.unique(y_true)

    # Calculate percentages
    if percent_type == "row":
        sums = cm.sum(axis=1, keepdims=True)
        perc = cm / np.where(sums == 0, 1, sums) * 100
        perc_label = "Row %"
    elif percent_type == "column":
        sums = cm.sum(axis=0, keepdims=True)
        perc = cm / np.where(sums == 0, 1, sums) * 100
        perc_label = "Col %"
    elif percent_type == "all":
        total = cm.sum()
        perc = (cm / total * 100) if total != 0 else np.zeros_like(cm, dtype=float)
        perc_label = "All %"
    else:
        raise ValueError("percent_type must be 'row', 'column', or 'all'")

    # Annotation with count and percentage
    annot = np.empty_like(cm).astype(str)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            annot[i, j] = f"{cm[i, j]}\n({perc[i, j]:.0f}%)"

    fig, ax = plt.subplots(figsize=figsize)

    # ✅ Heatmap uses PERCENTAGES so colors + colorbar are %
    im = ax.imshow(perc, cmap=cmap, vmin=0, vmax=100)
    cbar = fig.colorbar(im, ax=ax)
    cbar.ax.tick_params(labelsize=13)
    cbar.set_label("Percent (%)", fontsize=13)

    # Show numbers
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            if text_color is None:
                # background is % (0..100), so use 50 as midpoint
                cell_color = "black" if perc[i, j] > 50 else "white"
            else:
                cell_color = text_color

            ax.text(
                j, i, annot[i, j],
                ha="center", va="center",
                fontsize=13,
                color=cell_color
            )

    ax.set(
        xticks=np.arange(len(labels)),
        yticks=np.arange(len(labels)),
        xticklabels=labels,
        yticklabels=labels,
        xlabel="Predicted label",
        ylabel="True label",
    )

    ax.set_xlabel("Predicted label", fontsize=14)
    ax.set_ylabel("True label", fontsize=14)
    ax.tick_params(axis="x", labelsize=13)
    ax.tick_params(axis="y", labelsize=13)

    full_title = title if title else f"Confusion Matrix with {perc_label}"
    ax.set_title(full_title, fontsize=14)

    plt.tight_layout()
    if save_path is not None:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()


In [8]:
def custom_score(y_true, y_pred, group_ids, threshold, eval=False):
    '''
    Custom score function to correct early detections to correct
    '''
    # Binary prediction
    y_bin = (y_pred >= threshold).astype(int)
    print(f"f1 before mod: {f1_score(y_true, y_bin)}")
    # Create working DataFrame
    df = pd.DataFrame({
        'henkilotunnus': group_ids,
        'true_label': y_true,
        'predicted_label': y_bin
    })

    df['misclassified'] = (df['true_label'] != df['predicted_label']).astype(int)
    df['misclassified_proximity'] = 0

    # Set proximity rule
    for name, group in df.groupby('henkilotunnus'):
        for i in range(len(group) - 1):
            row = group.iloc[i]
            next_row = group.iloc[i + 1]

            if (
                row['misclassified'] == 1
                and row['true_label'] == 0
                and row['predicted_label'] == 1
                and next_row['true_label'] == 1
            ):
                df.loc[group.index[i], 'misclassified_proximity'] = 1
 
    # Adjust predictions
    adjusted_pred = df['predicted_label'].copy()
    mask = (df['misclassified'] == 1) & (df['misclassified_proximity'] == 1)
    adjusted_pred[mask] = df['true_label'][mask]

    if eval:
        return df, f1_score(df['true_label'], adjusted_pred), adjusted_pred
    else:
        return f1_score(df['true_label'], adjusted_pred)

In [9]:
def move_column_inplace(df, col, pos):
    col = df.pop(col)
    df.insert(pos, col.name, col)

## Data

In [10]:
SEED = 43
os.environ["PYTHONHASHSEED"] = str(SEED)
np.random.seed(SEED)

In [ ]:
# different data imputation potions
dfnames = ['interpolation', 'uncertainty', 'NOimputation']
i = 2
DATA_SOURCE = dfnames[i]
PLOT_PATH = f'/path/to/plots/train2_test2/{DATA_SOURCE}/'
DATA_PATH = f"/path/to/data/train2_test2/{DATA_SOURCE}/"
print("Data source:", DATA_SOURCE)

In [12]:
# load data
df = pd.read_csv(f"/path/to/data/model_data_{DATA_SOURCE}.csv")
df_other = df.copy()
df = df.drop(columns=['Unnamed: 0', 'ab_max_5d', 'has_ab', "bneut_accumulated_max_050_cycle", 'bneut_accumulated_max_050_total', "bneut_accumulated_max_005_total", "bneut_accumulated_max_005_cycle"])
rmv_list = df.loc[df['fn_day']==1].index # remove day when febrile neutropenia occurs
df = df.drop(rmv_list)

X = df.drop(columns=['henkilotunnus', 'infektion_binary', 'treatment_date', 'naytteenotto_hetki', 'fn_day'])
y = df['infektion_binary']
group = df['henkilotunnus']


In [14]:
# save featurenames as list for later usage
pd.DataFrame(X.columns.to_numpy(), columns=['Feature']).to_csv(DATA_PATH + "X_features_as_list.csv")

### Statistics from dataset

In [ ]:
print('Number of patients:', len(df['henkilotunnus'].drop_duplicates()))
print('Number of treatment cycles:', len(df.groupby(['henkilotunnus', 'cycle_number']).size()))
print('Number of datapoints:', len(df))
print()
print('Number of features:', len(X.columns))
print("Number of Inductions (row %):", round(len(X[X['sykli_IND']==1])/len(X), 3))
print("Number of Consolidations (row %):", round(len(X[X['sykli_IND']==0])/len(X), 3))
print("Number of treatment cycles:", len(df.groupby(['henkilotunnus', 'cycle_number']).size()))
print()
print("Number of Induction cycles:", len(df[df['sykli_IND']==1].groupby(['henkilotunnus', 'cycle_number']).size()))
print("Number of Consolidation cycles:", len(df[df['sykli_IND']==0].groupby(['henkilotunnus', 'cycle_number']).size()))
print()
print("Number of infection cycles in Inductions:", df[df['sykli_IND']==1].groupby(['henkilotunnus', 'cycle_number'])['infektion_binary'].any().sum())
print("Number of infection cycles in Consolidations:", df[df['sykli_IND']==0].groupby(['henkilotunnus', 'cycle_number'])['infektion_binary'].any().sum())
print('Number of FN:', len(df[df['infektion_binary']==1]))
print('Number of non-FN:', len(df[df['infektion_binary']==0]))
print()
years = pd.DatetimeIndex(df['naytteenotto_hetki']).year.drop_duplicates().sort_values()
print(f"Data collected between years {years.min()}-{years.max()}")

### Split the data into train / test / validation
- Group-wise
- Test 20%, train 80%

In [16]:
# Group depending test / train split
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
for train_idx, test_idx in splitter.split(X, y, groups=group):
    X_tr, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_tr, y_test = y.iloc[train_idx], y.iloc[test_idx]
    group_tr, group_test = group.iloc[train_idx], group.iloc[test_idx]
    df_tr, df_test = df.iloc[train_idx], df.iloc[test_idx]


Remove not needed lab value columns

In [17]:
dfs = [X_test, X_tr, df_test]

drop_nuisances = False
if drop_nuisances:
    for i, df in enumerate(dfs):
        cols_to_drop = [
            c for c in df.columns
            if c.startswith("s_") or (c.startswith("p_") and not c.startswith("p_crp"))
        ]
        df.drop(columns=cols_to_drop, inplace=True)


In [ ]:
print('Number of features:', len(X_tr.columns))

In [19]:
# save test train splits for later use
X_tr.to_csv(DATA_PATH + "X_train.csv")
y_tr.to_csv(DATA_PATH + "y_train.csv")
X_test.to_csv(DATA_PATH + "X_test.csv")
y_test.to_csv(DATA_PATH + "y_test.csv")
group_tr.to_csv(DATA_PATH + "group_train.csv")
group_test.to_csv(DATA_PATH + "group_test.csv")
df_test.to_csv(DATA_PATH + "df_test.csv")
df_tr.to_csv(DATA_PATH + 'df_tr.csv')

In [ ]:
# more statistics after split
nofones = y.tolist().count(1)/y.tolist().count(0)
print(f"Portion of 1 in dataset: {nofones:.3f}")
print()
print('Number of induction datapoints:', len(X[X['sykli_IND']==1]))
print('Number of consolidation datapoints:', len(X[X['sykli_IND']==0]))
print()
print('Number of pateints in train set', len(df_tr['henkilotunnus'].drop_duplicates()))
print('Number of pateints in test set', len(df_test['henkilotunnus'].drop_duplicates()))
print()
print('Number of treatment cycles in train set:', len(df_tr.groupby(['henkilotunnus', 'cycle_number']).size()))
print('Number of treatment cycles in test set:', len(df_test.groupby(['henkilotunnus', 'cycle_number']).size()))
print()
print('Number of datapoints in train set:', len(df_tr))
print('Number of datapoints in test set:', len(df_test))
print()
print("Number of FN in training set", int(y_tr.reset_index()['infektion_binary'].sum()))
print("Number of FN in testing set", int(y_test.reset_index()['infektion_binary'].sum()))

### Regression data

Writes `df_regression_tr.csv` / `df_regression_test.csv`, which the SHAP top-variables notebook reads.

In [23]:
df_other_tr = df_other[df_other['henkilotunnus'].isin(group_tr)][['henkilotunnus', 'naytteenotto_hetki', 'cycle_number', 'infektion_binary', "fn_day", "temperature"]]
df_other_test = df_other[df_other['henkilotunnus'].isin(group_test)][['henkilotunnus', 'naytteenotto_hetki', 'cycle_number', 'infektion_binary', "fn_day", "temperature"]]
df_other_test["naytteenotto_hetki"] = pd.to_datetime(df_other_test["naytteenotto_hetki"])

# FN date for each (patient, cycle)
fn_date = (
    df_other_test.loc[df_other_test["fn_day"] == 1]
      .groupby(["henkilotunnus", "cycle_number"])["naytteenotto_hetki"]
      .max()
      .rename("fn_date")
)

df_other_test = df_other_test.merge(fn_date, on=["henkilotunnus", "cycle_number"], how="left")

days_before = (df_other_test["fn_date"] - df_other_test["naytteenotto_hetki"]).dt.days

df_other_test["event"] = 0
df_other_test.loc[days_before.eq(1), "event"] = 2          # 24 h before FN
df_other_test.loc[days_before.eq(2), "event"] = 1         # 48 h before FN
df_other_test.loc[df_other_test["fn_day"].eq(1), "event"] = 3         # FN day itself (0 h)


df_other_test = df_other_test[df_other_test["event"]>0]
df_other_tr["naytteenotto_hetki"] = pd.to_datetime(df_other_tr["naytteenotto_hetki"])

# FN date for each (patient, cycle)
fn_date = (
    df_other_tr.loc[df_other_tr["fn_day"] == 1]
      .groupby(["henkilotunnus", "cycle_number"])["naytteenotto_hetki"]
      .max()
      .rename("fn_date")
)

df_other_tr = df_other_tr.merge(fn_date, on=["henkilotunnus", "cycle_number"], how="left")

days_before = (df_other_tr["fn_date"] - df_other_tr["naytteenotto_hetki"]).dt.days

df_other_tr["event"] = 0
df_other_tr.loc[days_before.eq(1), "event"] = 2          # 24 h before FN
df_other_tr.loc[days_before.eq(2), "event"] = 1         # 48 h before FN
df_other_tr.loc[df_other_tr["fn_day"].eq(1), "event"] = 3         # FN day itself (0 h)

df_other_tr = df_other_tr[df_other_tr["event"]>0]

# save for later use
df_other_tr.to_csv(DATA_PATH + 'df_regression_tr.csv')
df_other_test.to_csv(DATA_PATH + 'df_regression_test.csv')




## Grid search hyperparameter tuning

In [25]:
# 2) Imbalance scale from TRAIN
neg_cnt = y_tr.value_counts()[0]
pos_cnt = y_tr.value_counts()[1]
scale1 = (neg_cnt / pos_cnt) ** 0.5
scale2 = (neg_cnt / pos_cnt)

In [26]:
GS = False
if GS:    

    # 1) Train/val split with groups (your style)
    splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
    for train_idx, val_idx in splitter.split(X_tr, y_tr, groups=group_tr):
        X_train, X_val = X_tr.iloc[train_idx], X_tr.iloc[val_idx]
        y_train, y_val = y_tr.iloc[train_idx], y_tr.iloc[val_idx]
        group_train, group_val = group_tr.iloc[train_idx], group_tr.iloc[val_idx]

    # 2) Imbalance scale from TRAIN
    neg_cnt = y_train.value_counts()[0]
    pos_cnt = y_train.value_counts()[1]
    scale1 = (neg_cnt / pos_cnt) ** 0.5
    scale2 = (neg_cnt / pos_cnt)

    # 3) Base model – NOTE early_stopping_rounds HERE, not in fit
    xgb = XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        #eval_metric ='aucpr',
        use_label_encoder=False,
        random_state=SEED,
        tree_method="hist",
        early_stopping_rounds=10,      # <-- moved here
        seed=SEED,
    )

    # 4) Param grid
    param_grid = {
        "max_depth": [4, 5, 6],
        "n_estimators": [90, 100],
        "learning_rate": [0.05, 0.1],
        "subsample": [0.7, 0.8, 0.9, 1.0],
        "colsample_bytree": [0.7, 0.8, 0.9],
        "scale_pos_weight": [1.0, scale1, scale2],  # fixed name
    }

    cv = GroupKFold(n_splits=3)
    cv = StratifiedGroupKFold(n_splits=3)

    grid_search = GridSearchCV(
        estimator=xgb,
        param_grid=param_grid,
        scoring="neg_log_loss",
        #scoring="average_precision",
        cv=cv,
        n_jobs=-1,
        verbose=1,
    )

    # 5) Fit grid search – NO early_stopping_rounds here anymore
    grid_search.fit(
        X_train,
        y_train,
        groups=group_train,
        eval_set=[(X_val, y_val)],
        verbose=False,
    )

    print("Best params (grid):", grid_search.best_params_)
    print("Best CV score (grid):", grid_search.best_score_)

    best_xgb_grid = grid_search.best_estimator_


number we want to divide amount of false positives when calculating F1-score
- This decreases the influence of amount of false positives when calculating F1-score and makes the amount of false positives more influence

In [27]:
MULTI = 3

Functions used in threshold tuning

In [ ]:
def custom_cost_3fp_fn(y_true, y_pred, c_fp=3.0, c_fn=1.0):
    """Cost = c_fp * FP + c_fn * FN (defaults: 3*FP + 1*FN)."""
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    # return c_fp * fp + c_fn * fn
    return fp/c_fp + c_fn * fn

def custom_f1(y_true, y_pred, multi=MULTI):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    fp_adj = fp / multi
    precision = tp / (tp + fp_adj) if (tp + fp_adj) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    return 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0


In [29]:
# hyperparams
n_folds = 5
thresholds = np.linspace(0, 1, 100)


In [30]:
def _fp_fn_for_preds(y_true, y_pred):
    _, fp, fn, _ = confusion_matrix(y_true, y_pred).ravel()
    return fp, fn

def threshold_metrics(y_true, y_prob, thresholds=np.linspace(0, 1, 100), c_fp=3.0, c_fn=1.0):
    f1s, custom_f1s, custom_f13, precisions, recalls, costs = [], [], [], [], [], []
    fps, fns = [], []
    for thresh in thresholds:
        y_pred = (y_prob > thresh).astype(int)

        f1s.append(f1_score(y_true, y_pred, zero_division=0))
        custom_f1s.append(custom_f1(y_true, y_pred))            # your existing
        custom_f13.append(custom_f1(y_true, y_pred, multi=MULTI))   # your existing
        precisions.append(precision_score(y_true, y_pred, zero_division=0))
        recalls.append(recall_score(y_true, y_pred, zero_division=0))

        fp, fn = _fp_fn_for_preds(y_true, y_pred)
        fps.append(fp); fns.append(fn)
        costs.append(3.0 * fp + 1.0 * fn)  # or c_fp/c_fn

    return (np.array(f1s), np.array(custom_f1s), np.array(custom_f13),
            np.array(precisions), np.array(recalls),
            np.array(costs), np.array(fps), np.array(fns))


### N estimators

In [ ]:
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
for train_idx, val_idx in splitter.split(X_tr, y_tr, groups=group_tr):

    X_train, X_val = X_tr.iloc[train_idx], X_tr.iloc[val_idx]
    y_train, y_val = y_tr.iloc[train_idx], y_tr.iloc[val_idx]
    group_train, group_val = group_tr.iloc[train_idx], group_tr.iloc[val_idx]
 
# Make sure eval_metric is set; set n_estimators high and rely on early stopping
if i==0:
    preset = {
        'colsample_bytree': 0.9,
        'learning_rate': 0.05,
        'max_depth': 4,
        'n_estimators': 2000,
        'scale_pos_weigth': scale2,
        'subsample': 0.8,
        "eval_metric":'logloss',
        "use_label_encoder":False,
        "verbosity":0,
        "objective":'binary:logistic', # pitäsköhä tähä tehä joku uus? 
        "early_stopping_rounds":10,
        }
elif i==1:
    preset = { 
        'colsample_bytree': 0.8,
        'learning_rate': 0.05,
        'max_depth': 5,
        'n_estimators': 2000,
        'scale_pos_weigth': scale2,
        'subsample': 0.7,
        "eval_metric":'logloss',
        "use_label_encoder":False,
        "verbosity":0,
        "objective":'binary:logistic', # pitäsköhä tähä tehä joku uus? 
        "early_stopping_rounds":10,
        }
else:
        preset = {
        'colsample_bytree': 0.9,
        'learning_rate': 0.05,
        'max_depth': 4,
        'n_estimators': 2000,
        'scale_pos_weigth': 1,
        'subsample': 0.9,
        "eval_metric":'logloss',
        "use_label_encoder":False,
        "verbosity":0,
        "objective":'binary:logistic', # pitäsköhä tähä tehä joku uus? 
        "early_stopping_rounds":10,
        }

model = XGBClassifier(**preset, seed=SEED, random_state=SEED)

model.fit(
    X_train.values, y_train.values,
    eval_set=[(X_train.values, y_train.values),
              (X_val.values,   y_val.values)],
    verbose=False
)

# --- Plot learning curves ---
results = model.evals_result()
train_loss = results['validation_0']['logloss']   # first tuple in eval_set
val_loss   = results['validation_1']['logloss']   # second tuple in eval_set
epochs = range(1, len(train_loss) + 1)

plt.figure()
plt.plot(epochs, train_loss, label='train logloss')
plt.plot(epochs, val_loss,   label='val logloss')
plt.axvline(model.best_iteration+1, linestyle='--',
            label=f'best iteration = {model.best_iteration+1}')
plt.xlabel('Number of trees')
plt.ylabel('Logloss')
plt.title('XGBoost learning curves')
plt.legend()
plt.show()
plt.savefig(PLOT_PATH+'/train/n_estimators_all_variables.png', dpi=300, bbox_inches="tight")

print("Best iteration:", model.best_iteration+1)
best_n_estimates = model.best_iteration+1


 for each dataset best hyperparams

In [32]:
if i==0:
    preset = {
        'colsample_bytree': 0.9,
        'learning_rate': 0.05,
        'max_depth': 4,
        'n_estimators': min(best_n_estimates, 100),
        'scale_pos_weigth': scale2,
        'subsample': 0.8,
        "eval_metric":'logloss',
        "use_label_encoder":False,
        "verbosity":0,
        "objective":'binary:logistic', # pitäsköhä tähä tehä joku uus? 
        "early_stopping_rounds":10,
        }
elif i==1:
    preset = {
        'colsample_bytree': 0.8,
        'learning_rate': 0.05,
        'max_depth': 5,
        'n_estimators': min(best_n_estimates, 100),
        'scale_pos_weigth': scale2,
        'subsample': 0.7,
        "eval_metric":'logloss',
        "use_label_encoder":False,
        "verbosity":0,
        "objective":'binary:logistic', # pitäsköhä tähä tehä joku uus? 
        "early_stopping_rounds":10,
    }
else:
    preset = {
        'colsample_bytree': 0.9,
        'learning_rate': 0.05,
        'max_depth': 4,
        'n_estimators': min(best_n_estimates, 100),
        'scale_pos_weigth': 1,
        'subsample': 0.9,
        "eval_metric":'logloss',
        "use_label_encoder":False,
        "verbosity":0,
        "objective":'binary:logistic', # pitäsköhä tähä tehä joku uus? 
        "early_stopping_rounds":10,
    } 


## Model optimization
- Threshold tuning

### Threshold tuning

Selects the threshold that satisfies FP > MULTI·FN.

In [33]:
OVERSAMPLE = False  # possible to try oversampling using different methods
USE_PRESET = True   # possible to use pre set hyperparameters or start from empty model

In [34]:


def threshold_over_sampling(
        X_tr, y_tr, group_tr,
        multi=MULTI,
        n_folds=5,
        ratio='auto'
        ):
    thresholds = np.linspace(0, 1, 100)

    all_f1s, all_custom_f1s, all_F1_3s = [], [], []
    all_precisions, all_recalls, all_costs, all_fps, all_fns = [], [], [], [], []

    gkf = StratifiedGroupKFold(n_splits=n_folds, shuffle=True, random_state=SEED)


    for train_idx, val_idx in gkf.split(X_tr, y_tr, groups=group_tr):  # your group var
        X_train, X_val = X_tr.iloc[train_idx], X_tr.iloc[val_idx]
        y_train, y_val = y_tr.iloc[train_idx], y_tr.iloc[val_idx]

        if OVERSAMPLE:    
            # --- NEW: oversample train only ---
            ros = RandomOverSampler(
                sampling_strategy=ratio,   # or e.g. 0.3, {1: 5000}, ...
                random_state=SEED,
            )
            X_train_res, y_train_res = ros.fit_resample(X_train, y_train)
        else:
            X_train_res = X_train
            y_train_res = y_train

        # --- XGBoost on oversampled data ---
        if USE_PRESET:
            model = XGBClassifier(**preset, seed=SEED, random_state=SEED)
        else:
            model = XGBClassifier(early_stopping_rounds=10, seed=SEED, random_state=SEED)

        model.fit(
            X_train_res.values, y_train_res.values,
            eval_set=[(X_val.values, y_val.values)],   # val set is NOT oversampled
            verbose=False,
        )

        # --- same threshold tuning as before ---
        y_prob = model.predict_proba(X_val.values)[:, 1]

        f1s, custom_f1s, F1_3s, precisions, recalls, costs, fps, fns = \
            threshold_metrics(y_val.values, y_prob, thresholds)

        all_f1s.append(f1s)
        all_F1_3s.append(F1_3s)
        all_precisions.append(precisions)
        all_recalls.append(recalls)
        all_costs.append(costs)
        all_fps.append(fps)
        all_fns.append(fns)

    # then your aggregation code:
    all_f1s        = np.array(all_f1s)
    all_F1_3s      = np.array(all_F1_3s)
    all_precisions = np.array(all_precisions)
    all_recalls    = np.array(all_recalls)
    all_costs      = np.array(all_costs)
    all_fps        = np.array(all_fps)
    all_fns        = np.array(all_fns)

    mean_f1        = all_f1s.mean(axis=0)
    mean_F1_3      = all_F1_3s.mean(axis=0)
    mean_precision = all_precisions.mean(axis=0)
    mean_recall    = all_recalls.mean(axis=0)
    mean_costs     = all_costs.mean(axis=0)
    mean_fp        = all_fps.mean(axis=0)
    mean_fn        = all_fns.mean(axis=0)
    
    # ... your feasible-mask + plotting code exactly as in screenshot
    feasible = mean_fp <= multi * mean_fn
    print(feasible)

    if feasible.any():

        # select lower limit
        nz = np.flatnonzero(~feasible)
        idx_best_constrained = int(nz[-1])
        print(idx_best_constrained)

    else:
        # If nothing feasible, fall back to the most conservative threshold (highest thr)
        idx_best_constrained = 0.5

    thresh_best_constrained = thresholds[idx_best_constrained]

    fig, ax1 = plt.subplots(figsize=(10, 6))

    # Left axis: scores in [0,1]
    ax1.plot(thresholds, mean_f1,        label='F1')
    ax1.plot(thresholds, mean_F1_3,      label='Custom F1 (FP/3)')
    ax1.plot(thresholds, mean_precision, label='Precision')
    ax1.plot(thresholds, mean_recall,    label='Recall')
    ax1.set_xlabel('Threshold', fontsize=15); 
    ax1.set_ylabel('Score', fontsize=15); 
    ax1.tick_params(axis='both', labelsize=14)
    ax1.set_ylim(0, 1); 
    ax1.grid(True)

    # Shade infeasible region where FP > 3·FN
    ax1.fill_between(thresholds, 0, 1, where=~feasible, alpha=0.12, transform=ax1.get_xaxis_transform(),
                    label=f'Infeasible (FP > {multi}·FN)')

    # Mark constrained-best threshold

    ax1.axvline(thresh_best_constrained, color='crimson', linestyle='--', linewidth=1)

    # Legends merged
    lines1, labels1 = ax1.get_legend_handles_labels()
    #lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1, labels1, loc='best', fontsize=13)
    ax1.set_ylim(0, 1)
    ax1.set_yticks(np.arange(0, 1.01, 0.1))
    ax1.grid(True)

    # Simple annotations
    def annotate_with_arrow(ax, x, y, text, dx=0.02, dy=0.02, fmt="{:.3f}"):
        ax.annotate(
            f"{text}\n{fmt.format(x)}",
            xy=(x, y),
            xytext=(x + dx, y + dy),
            arrowprops=dict(arrowstyle='->'),
            bbox=dict(boxstyle='round', fc='w'),
        )

    annotate_with_arrow(ax1, thresh_best_constrained,  idx_best_constrained, 'Constraint best')             
    plt.tight_layout(); plt.show()
    plt.savefig(PLOT_PATH+'/train/threshold_all_variables.png', dpi=300, bbox_inches="tight")


    return thresh_best_constrained


In [ ]:
if OVERSAMPLE:
    ratios = [0.2, 0.3, 0.4]
else:
    ratios = [None]

trs = []
for r in ratios:
    tr = threshold_over_sampling(
        X_tr, y_tr, group_tr,
        multi=MULTI,
        n_folds=5,
        ratio=r,
        )
    trs.append(tr)
    
trs

# Decision threshold used from here on. `threshold_over_sampling` picks it by
# 5-fold CV; with OVERSAMPLE = False there is a single ratio, so trs has one entry.
THRESHOLD = round(trs[0], 2)
print("Optimal threshold:", THRESHOLD)


## Actual model training

Whole data training before test

In [ ]:
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)

for train_idx, val_idx in splitter.split(X_tr, y_tr, groups=group_tr):
    X_train, X_val = X_tr.iloc[train_idx], X_tr.iloc[val_idx]
    y_train, y_val = y_tr.iloc[train_idx], y_tr.iloc[val_idx]

    if OVERSAMPLE:
    # ----- NEW: oversample training set only -----
        ros = RandomOverSampler(
            sampling_strategy="auto",   # CV ratio sweep above is disabled
            random_state=SEED,
        )
        X_train_res, y_train_res = ros.fit_resample(X_train, y_train)
    else:
        X_train_res = X_train
        y_train_res = y_train
    print(
        f"X_train: {X_train_res.shape}, y_train: {y_train_res.shape}, "
        f"X_val: {X_val.shape}, y_val: {y_val.shape}"
    )

    if USE_PRESET:
        best_model = XGBClassifier(**preset, seed=SEED)
    else:
        best_model = XGBClassifier(early_stopping_rounds=10)
    #best_model = XGBClassifier()

    best_model.fit(
        X_train_res.values,
        y_train_res.values,
        eval_set=[(X_train_res.values, y_train_res.values),
                  (X_val.values,       y_val.values)],
        verbose=False,
    )

# use original (un-oversampled) val set for metrics / threshold tuning
y_pred  = best_model.predict(X_val.values)
y_proba = best_model.predict_proba(X_val.values)[:, 1]


save model

In [41]:
# load the model for later use
# save the model (JSON or binary)
best_model.save_model(DATA_PATH + "xgb_model.json")
# save the feature names so R can pick them up
with open(DATA_PATH + "features.json", "w") as f:
    json.dump(list(X_train.columns), f)

## Training evaluation

In [ ]:
y_prob_train = best_model.predict_proba(X_train)[:, 1]

print("CM with custom threshold:")
y_pred_thresh = (y_prob_train >= THRESHOLD).astype(int)

plot_confusion_matrix_with_percentages(
    y_train, y_pred_thresh, 
    percent_type='row', 
    threshold=None,
    save_path= PLOT_PATH+'train/cm_mod_tr',
    title='Training set',
    cmap=colourmap,
    text_color='white'
    )

kappa = cohen_kappa_score(y_train, y_pred_thresh)
recall = recall_score(y_train, y_pred_thresh)
precision = precision_score(y_train, y_pred_thresh)
f1 = f1_score(y_train, y_pred_thresh)
accuracy = accuracy_score(y_train, y_pred_thresh)
customf1 = custom_f1(y_train, y_pred_thresh)
spec = recall_score(y_train, y_pred_thresh, pos_label=0)
pnv = calc_pnv(y_train, y_pred_thresh)


row = pd.DataFrame({'data':[f'training'], 'F1_score':[f1], 'kappa':[kappa], 'recall':[recall], 'precision':[precision], 'accuracy':[accuracy], 'customf1':[customf1], 'specificity':[spec], 'PNV':[pnv]})
metrics_df = pd.concat([metrics_df, row], ignore_index=True)
metrics_df

print('Kappa:', kappa) # Is my model better than random, and by how much?
print("Recall:", recall)
print("Precision:", precision)
print("F1 score:", f1)
print(f"Custom F1: {customf1:.4f}")


In [ ]:
def _row_metrics(y, yh):
    return {
        "F1_score":    f1_score(y, yh, zero_division=0),
        "kappa":       cohen_kappa_score(y, yh),
        "recall":      recall_score(y, yh, zero_division=0),
        "precision":   precision_score(y, yh, zero_division=0),
        "accuracy":    accuracy_score(y, yh),
        "customf1":    custom_f1(y, yh),
        "specificity": recall_score(y, yh, pos_label=0, zero_division=0),
        "PNV":         calc_pnv(y, yh),
    }

def add_metrics_column(table, y_true, y_prob, thr, name, groups=None, extra=None,
                       n_boot=2000, seed=42, alpha=0.05, add_auc=True):
    """Append one run to the metrics table as the column group (name, est/lo/hi).
       table=None starts a new table. Re-using a name replaces that run."""
    y = np.asarray(y_true).astype(int); p = np.asarray(y_prob, float)

    def _all(yy, pp):
        m = _row_metrics(yy, (pp >= thr).astype(int))
        if add_auc:
            m = {"AUROC": roc_auc_score(yy, pp),
                 "AUPRC": average_precision_score(yy, pp), **m}
        if extra:
            for k, fn_ in extra.items():
                m[k] = fn_(yy, (pp >= thr).astype(int))
        return m

    point = _all(y, p)
    keys = list(point)

    g = np.arange(len(y)) if groups is None else np.asarray(groups)
    _, inv = np.unique(g, return_inverse=True)
    idx = [np.flatnonzero(inv == k) for k in range(inv.max() + 1)]

    rng = np.random.default_rng(seed)
    boots = np.full((n_boot, len(keys)), np.nan)
    for b in range(n_boot):
        s = rng.integers(0, len(idx), len(idx))
        ii = np.concatenate([idx[j] for j in s])
        yb, pb = y[ii], p[ii]
        if yb.min() == yb.max():
            continue
        m = _all(yb, pb)
        boots[b] = [m[k] for k in keys]

    lo_q, hi_q = 100 * alpha / 2, 100 * (1 - alpha / 2)
    col = pd.DataFrame(
        {(name, "est"): [point[k] for k in keys],
         (name, "lo"):  np.nanpercentile(boots, lo_q, axis=0),
         (name, "hi"):  np.nanpercentile(boots, hi_q, axis=0)},
        index=pd.Index(keys, name="metric"))
    col.columns = pd.MultiIndex.from_tuples(col.columns, names=["run", "stat"])

    if table is None:
        return col
    if name in table.columns.get_level_values("run"):
        table = table.drop(columns=name, level="run")
    order = list(table.index) + [k for k in col.index if k not in table.index]
    return table.join(col, how="outer").reindex(order)


def fmt_table(df, decimals=2):
    """Collapse each run's est/lo/hi into one 'est (lo–hi)' column."""
    out = pd.DataFrame(index=df.index)
    for r in df.columns.get_level_values("run").unique():
        e, lo, hi = df[(r, "est")], df[(r, "lo")], df[(r, "hi")]
        out[r] = [f"{a:.{decimals}f} ({b:.{decimals}f}–{c:.{decimals}f})"
                  if pd.notna(a) else "" for a, b, c in zip(e, lo, hi)]
    return out

metrics_df_ci = add_metrics_column(None, y_train, y_prob_train,  THRESHOLD, "train_all")
fmt_table(metrics_df_ci)

### AUROC and AUPRC with 95% CI

In [ ]:
#y_train, y_prob_train
DCA_ORANGE = "#d95f02"
DCA_BLUE = "#2b6cb0"
DCA_BAND = "#fde68a"
DCA_GRAY = "#6b7280"
DCA_INK = "#111827"


def plot_curve_ci(y_true, y_prob, kind="roc", groups=None, n_boot=1000,
                  seed=42, alpha=0.05, n_grid=201, ax=None, title=None,
                  figsize=(7.0, 7.0), pos="upper right"):
    """ROC or precision-recall curve with a bootstrap 95% band and AUC (95% CI)."""
    y = np.asarray(y_true).astype(int); p = np.asarray(y_prob, float)
    grid = np.linspace(0, 1, n_grid)

    def _curve(yy, pp):
        if kind == "roc":
            fpr, tpr, _ = roc_curve(yy, pp)
            return np.interp(grid, fpr, tpr), roc_auc_score(yy, pp)
        pr, rc, _ = precision_recall_curve(yy, pp)
        return np.interp(grid, rc[::-1], pr[::-1]), average_precision_score(yy, pp)

    y_pt, auc_pt = _curve(y, p)

    g = np.arange(len(y)) if groups is None else np.asarray(groups)
    _, inv = np.unique(g, return_inverse=True)
    idx = [np.flatnonzero(inv == k) for k in range(inv.max() + 1)]

    rng = np.random.default_rng(seed)
    curves = np.full((n_boot, n_grid), np.nan)
    aucs = np.full(n_boot, np.nan)
    for b in range(n_boot):
        s = rng.integers(0, len(idx), len(idx))
        ii = np.concatenate([idx[j] for j in s])
        yb, pb = y[ii], p[ii]
        if yb.min() == yb.max():
            continue
        curves[b], aucs[b] = _curve(yb, pb)

    lo_q, hi_q = 100 * alpha / 2, 100 * (1 - alpha / 2)
    band_lo = np.nanpercentile(curves, lo_q, axis=0)
    band_hi = np.nanpercentile(curves, hi_q, axis=0)
    a_lo, a_hi = np.nanpercentile(aucs, lo_q), np.nanpercentile(aucs, hi_q)

    if ax is None:
        fig, ax = plt.subplots(figsize=figsize)
    else:
        fig = ax.figure

    name = "AUROC" if kind == "roc" else "AUPRC"
    ax.fill_between(grid, band_lo, band_hi, color=DCA_BLUE, alpha=0.20, lw=0, zorder=1)
    ax.plot(grid, y_pt, color=DCA_BLUE, lw=2, zorder=3)

    if kind == "roc":
        ax.plot([0, 1], [0, 1], color=DCA_GRAY, ls="--", lw=1, zorder=2)
        ax.set_xlabel("False positive rate", fontsize=14)
        ax.set_ylabel("True positive rate", fontsize=14)
        curve_lbl, loc = "ROC curve", "lower right"
    else:
        base = y.mean()
        ax.axhline(base, color=DCA_GRAY, ls="--", lw=1, zorder=2)
        ax.set_xlabel("Recall", fontsize=14)
        ax.set_ylabel("Precision", fontsize=14)
        curve_lbl, loc = "Precision–recall curve", pos

    _line = Line2D([], [], color=DCA_BLUE, lw=2)
    _band = Patch(facecolor=DCA_BLUE, alpha=0.20, lw=0)
    _txt  = Line2D([], [], ls="none", marker="none")
    _base = Line2D([], [], color=DCA_GRAY, ls="--", lw=1)

    ax.set_xlim(0, 1); ax.set_ylim(0, 1.02)
    ax.tick_params(labelsize=12)
    ax.grid(True, alpha=0.1)
    #ax.legend(fontsize=11, loc=loc, frameon=False)

    line = Line2D([], [], color=DCA_BLUE, lw=2)
    _band = Patch(facecolor=DCA_BLUE, alpha=0.20, lw=0)
    _txt  = Line2D([], [], ls="none", marker="none")     # text-only row
    _auc  = f"{name} = {auc_pt:.2f} (95% CI {a_lo:.2f}–{a_hi:.2f})"

    if kind == "roc":
        _h = [_line, _band, _txt]
        _l = [curve_lbl, "Curve – 95% CI", _auc]
    else:
        _base = Line2D([], [], color=DCA_GRAY, ls="--", lw=1)
        _h = [_line, _band, _txt]
        _l = [curve_lbl, "Curve – 95% CI", _auc]

    ax.legend(_h, _l, fontsize=11, loc=loc, frameon=False)
    ax.set_title(title or ("ROC curve" if kind == "roc" else "Precision-recall curve"),
                 fontsize=16)
    ax.set_box_aspect(1)
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    print(f"{name} {auc_pt:.3f} (95% CI {a_lo:.3f}-{a_hi:.3f})")
    return fig, (auc_pt, a_lo, a_hi)


fig, roc_ci  = plot_curve_ci(y_train, y_prob_train, kind="roc", groups=None)
fig.savefig(PLOT_PATH + 'all_train_ROC_curve_CI95', dpi=300, bbox_inches='tight')
plt.show()

fig, prc_ci = plot_curve_ci(y_train, y_prob_train, kind="pr", groups=None, pos="best")
fig.savefig(PLOT_PATH + 'all_train_RP_curve_CI95', dpi=300, bbox_inches='tight')
plt.show()

### One-day-early correction + custom threshold

In [ ]:
y_prob_train = best_model.predict_proba(X_train)[:, 1]

df, score, y_pred_mod = custom_score(y_train, y_prob_train, group_train , threshold=THRESHOLD, eval=True)

# Evaluation metrics
f1 = f1_score(y_train, y_pred_mod)
recall = recall_score(y_train, y_pred_mod)
accuracy = accuracy_score(y_train, y_pred_mod)
precision = precision_score(y_train, y_pred_mod)
kappa = cohen_kappa_score(y_train, y_pred_mod)
spec = recall_score(y_train, y_pred_mod, pos_label=0)
pnv = calc_pnv(y_train, y_pred_mod)

print(f"Kappa       : {kappa:.4f}")
print(f"F1 Score    : {f1:.4f}")
print(f"Recall      : {recall:.4f}")
print(f"Accuracy    : {accuracy:.4f}")
print(f"Precision   : {precision:.4f}")
customf1 = custom_f1(y_train, y_pred_mod)
print(f"Custom F1   : {customf1:.4f}")


# Confusion matrix
plot_confusion_matrix_with_percentages(
    y_train, y_pred_mod, 
    percent_type='row', 
    threshold=None,
    save_path= PLOT_PATH+'train/one_day_early_correction_cm',
    title=f'Training set\nConfusion Matrix with Row %\nthreshold={str(THRESHOLD)}',
    cmap='coolwarm',
    text_color='white'

)

row = pd.DataFrame({'data':[f'train_corrected'], 'F1_score':[f1], 'kappa':[kappa], 'recall':[recall], 'precision':[precision], 'accuracy':[accuracy], 'customf1':[customf1], 'specificity':[spec], 'PNV':[pnv]})
metrics_df = pd.concat([metrics_df, row], ignore_index=True)
metrics_df

metrics_df_ci = add_metrics_column(metrics_df_ci, y_train, y_pred_mod,  THRESHOLD, "train_all_corrected")
fmt_table(metrics_df_ci)


## Test

### Test set evaluation

In [ ]:
y_pred_test = best_model.predict(X_test)
print('Kappa:', cohen_kappa_score(y_test, y_pred_test))
print("Recall:", recall_score(y_test, y_pred_test))
print("Precision:", precision_score(y_test, y_pred_test))
print("F1 score:", f1_score(y_test, y_pred_test))
customf1 = custom_f1(y_test, y_pred_test)
print(f"Custom F1: {customf1:.4f}")

In [ ]:
y_prob_test = best_model.predict_proba(X_test)[:, 1]
print("CM with custom threshold:")

# Test confusion matrix using custom treshold
y_pred_thresh = (y_prob_test >= THRESHOLD).astype(int)

plot_confusion_matrix_with_percentages(
    y_test, y_pred_thresh, 
    percent_type='row', 
    threshold=None,
    save_path= PLOT_PATH+'test/cm_'+str(THRESHOLD).replace('.', '_'),
    title=f"Test set",
    cmap=colourmap,
    text_color='white'
)

kappa = cohen_kappa_score(y_test, y_pred_thresh)
recall = recall_score(y_test, y_pred_thresh)
precision = precision_score(y_test, y_pred_thresh)
f1 = f1_score(y_test, y_pred_thresh)
accuracy = accuracy_score(y_test, y_pred_thresh)
f1_mod = custom_f1(y_test, y_pred_thresh, multi=MULTI)
spec = recall_score(y_test, y_pred_thresh, pos_label=0)
pnv = calc_pnv(y_test, y_pred_thresh)
customf1 = custom_f1(y_test, y_pred_thresh)


row = pd.DataFrame({'data':[f'test'], 'F1_score':[f1], 'kappa':[kappa], 'recall':[recall], 'precision':[precision], 'accuracy':[accuracy], 'customf1':[customf1], 'specificity':[spec], 'PNV':[pnv]})
metrics_df = pd.concat([metrics_df, row], ignore_index=True)
metrics_df

print('Kappa:', kappa) # Is my model better than random, and by how much?
print("Recall:", recall)
print("Precision:", precision)
print("F1 score:", f1)
print("Accuracy:", accuracy)
print(f"Custom F1: {customf1:.4f}")

metrics_df_ci = add_metrics_column(metrics_df_ci, y_test, y_prob_test,  THRESHOLD, "test_all")
fmt_table(metrics_df_ci)


### AUROC and AUPRC with 95% CI

In [ ]:
# y_test, y_prob_test
fig, roc_ci  = plot_curve_ci(y_test, y_prob_test, kind="roc", groups=None)
fig.savefig(PLOT_PATH + 'all_test_ROC_curve_CI95', dpi=300, bbox_inches='tight')
plt.show()

fig, prc_ci = plot_curve_ci(y_test, y_prob_test, kind="pr", groups=None, pos="best")
fig.savefig(PLOT_PATH + 'all_test_RP_curve_CI95', dpi=300, bbox_inches='tight')
plt.show()

### One-day-early correction

In [ ]:
y_prob_test = best_model.predict_proba(X_test)[:, 1]
df, score, y_pred_mod = custom_score(y_test, y_prob_test, group_test, threshold=THRESHOLD, eval=True)
#df, score, y_pred_mod = custom_score(y_test, y_prob_test, group_test, threshold=0.04, eval=True)

# Evaluation metrics
f1 = f1_score(y_test, y_pred_mod)
recall = recall_score(y_test, y_pred_mod)
accuracy = accuracy_score(y_test, y_pred_mod)
precision = precision_score(y_test, y_pred_mod)
kappa = cohen_kappa_score(y_test, y_pred_mod)
customf1 = custom_f1(y_test, y_pred_mod)
spec = recall_score(y_test, y_pred_mod, pos_label=0)
pnv = calc_pnv(y_test, y_pred_mod)


print(f"Kappa       : {kappa:.4f}")
print(f"F1 Score    : {f1:.4f}")
print(f"Recall      : {recall:.4f}")
print(f"Accuracy    : {accuracy:.4f}")
print(f"Precision   : {precision:.4f}")
print(f"Custom F1   : {customf1:.4f}")


# Confusion matrix
plot_confusion_matrix_with_percentages(
    y_test, y_pred_mod, 
    percent_type='row', 
    threshold=None,
    save_path= PLOT_PATH+'test/one_day_early_correction/cm_mod_test',
    title=f"Test set\nConfusion matrix with Row %\nthreshold={str(THRESHOLD)}",
    cmap='coolwarm',
    text_color='white'
)

row = pd.DataFrame({'data':[f'test_corrected'], 'F1_score':[f1], 'kappa':[kappa], 'recall':[recall], 'precision':[precision], 'accuracy':[accuracy], 'customf1':[customf1], 'specificity':[spec], 'PNV':[pnv]})
metrics_df = pd.concat([metrics_df, row], ignore_index=True)
metrics_df

metrics_df_ci = add_metrics_column(metrics_df_ci, y_test, y_pred_mod,  THRESHOLD, "test_all_corrected")
fmt_table(metrics_df_ci)


In [52]:
# all metrics to same table
eval_df = df_test.copy()
eval_df['modified_y'] = y_pred_mod
eval_df['original_y'] = y_pred_thresh
move_column_inplace(eval_df, 'modified_y', 1)
move_column_inplace(eval_df, 'original_y', 1)
move_column_inplace(eval_df, 'naytteenotto_hetki', 5)

changed_df = eval_df.loc[eval_df['original_y'] != eval_df['modified_y']]
changed_df.to_csv(DATA_PATH + "one_day_early.csv")
#changed_df

### Induction evaluation

In [ ]:
# we can use earlier created eval_df
ind = eval_df.loc[eval_df['sykli_IND']==1]

#df, score, y_pred_mod = custom_score(ind['y_true'], y_prob_test, group_test, threshold=0.29, eval=True)
y_test = ind['infektion_binary']
y_pred_mod = ind['modified_y']
y_pred_ = ind['original_y']

# 👉 3. Evaluation metrics
f1 = f1_score(y_test, y_pred_mod)
recall = recall_score(y_test, y_pred_mod)
accuracy = accuracy_score(y_test, y_pred_mod)
precision = precision_score(y_test, y_pred_mod)

# build a metrics table
rows = pd.DataFrame({
    'data':['IND', 'IND_corrected'],
    'F1_score': [f1_score(y_test, y_pred_), f1_score(y_test, y_pred_mod)],
    'kappa' : [cohen_kappa_score(y_test, y_pred_), cohen_kappa_score(y_test, y_pred_mod)],
    'recall': [recall_score(y_test, y_pred_), recall_score(y_test, y_pred_mod)],
    'precision' : [precision_score(y_test, y_pred_), precision_score(y_test, y_pred_mod)],
    'accuracy' : [accuracy_score(y_test, y_pred_), accuracy_score(y_test, y_pred_mod)],
    'customf1' : [custom_f1(y_test, y_pred_), custom_f1(y_test, y_pred_mod)],
    'specificity':[recall_score(y_test, y_pred_, pos_label=0), recall_score(y_test, y_pred_mod, pos_label=0)],
    'PNV':[calc_pnv(y_test, y_pred_), calc_pnv(y_test, y_pred_mod)]

})
metrics_df = pd.concat([metrics_df, rows], ignore_index=True)

plot_confusion_matrix_with_percentages(
    y_test, y_pred_mod, 
    percent_type='row', 
    threshold=THRESHOLD,
    title='Induction: Confusion Matrix\nTest set',
    save_path=PLOT_PATH+'test/cm_induction_correction',
    cmap='coolwarm',
    text_color='white'
)

display(metrics_df.style.format({"Value": "{:.4f}"}))

metrics_df_ci = add_metrics_column(metrics_df_ci, y_test, y_pred_,  THRESHOLD, "ind_all")
metrics_df_ci = add_metrics_column(metrics_df_ci, y_test, y_pred_mod,  THRESHOLD, "ind_all_corrected")
fmt_table(metrics_df_ci)


### Consolidation evaluation

In [ ]:
kons = eval_df.loc[eval_df['sykli_IND']!=1]


#df, score, y_pred_mod = custom_score(kons['y_true'], y_prob_test, group_test, threshold=0.29, eval=True)
y_test = kons['infektion_binary']
y_pred_mod = kons['modified_y']
y_pred_ = kons['original_y']

# build a metrics table
rows = pd.DataFrame({
    'data':['KONS', 'KONS_corrected'],
    'F1_score': [f1_score(y_test, y_pred_), f1_score(y_test, y_pred_mod)],
    'kappa' : [cohen_kappa_score(y_test, y_pred_), cohen_kappa_score(y_test, y_pred_mod)],
    'recall': [recall_score(y_test, y_pred_), recall_score(y_test, y_pred_mod)],
    'precision' : [precision_score(y_test, y_pred_), precision_score(y_test, y_pred_mod)],
    'accuracy' : [accuracy_score(y_test, y_pred_), accuracy_score(y_test, y_pred_mod)],
    'customf1' : [custom_f1(y_test, y_pred_), custom_f1(y_test, y_pred_mod)],
    'specificity':[recall_score(y_test, y_pred_, pos_label=0), recall_score(y_test, y_pred_mod, pos_label=0)],
    'PNV':[calc_pnv(y_test, y_pred_), calc_pnv(y_test, y_pred_mod)]
})
metrics_df = pd.concat([metrics_df, rows], ignore_index=True)

plot_confusion_matrix_with_percentages(
    y_test, y_pred_mod, 
    percent_type='row', 
    threshold=THRESHOLD,
    title='Consolidation: Confusion Matrix\nTest set',
    save_path=PLOT_PATH+'test/cm_consolidation_correction',
    cmap='coolwarm',
    text_color='white'
)

display(metrics_df.style.format({"Value": "{:.4f}"}))

metrics_df_ci = add_metrics_column(metrics_df_ci, y_test, y_pred_,  THRESHOLD, "kons_all")
metrics_df_ci = add_metrics_column(metrics_df_ci, y_test, y_pred_mod,  THRESHOLD, "kons_all_corrected")
fmt_table(metrics_df_ci)

## Save metrics table

In [ ]:
metrics_df.to_csv(PLOT_PATH +'metrics_table.csv')
print(PLOT_PATH +'metrics_table.csv')

metrics_df_ci.to_csv(PLOT_PATH + "CI_metrics_table")
print(PLOT_PATH + "CI_metrics_table")